In [64]:
import ollama
import json

In [65]:
with open("data/grids.json", "r") as f:
    grids = json.load(f)

In [66]:
MODEL = "mistral"

In [67]:
SYSTEM_PROMPT = """
You are a helpful assistant that helps a robot navigate a grid world. 


Environment:
    Coordinate system:
        - A position is represented as [row, column].
        - The first number is the row.
        - The second number is the column.

    Action rules:
        - North: subtract 1 from the FIRST number of the position.
        - South: add 1 to the FIRST number of the position.
        - East: add 1 to the SECOND number of the position.
        - West: subtract 1 from the SECOND number of the position.

    The robot is placed in a 5x5 grid world:
        - The robot can only move to open positions.
        - The robot cannot move into a position of an obstacle.
        - The robot cannot move outside the grid boundaries.


Task:
    - Provide a path for the robot to reach the goal from the starting position while avoiding the blocked cells and staying within the grid boundaries.

Output:
    - Mention the position of the robot after each step.
    - Finally, after reaching the goal, provide an array of directions (North, South, East, West) that the robot should take to reach the goal from the starting position.
    - Do not include any additional text or explanations in your response.

#####################

Example 1:

User Prompt:
Starting position: [0, 0]
Goal position: [4, 4]
Obstacles: [[0, 1], [0, 3], [1, 2], [1, 3]]

Output:
    - Position: [0, 0] -> [1, 0] (South)
    - Position: [1, 0] -> [1, 1] (East)
    - Position: [1, 1] -> [2, 1] (South)
    - Position: [2, 1] -> [2, 2] (East)
    - Position: [2, 2] -> [3, 2] (South)
    - Position: [3, 2] -> [3, 3] (East)
    - Position: [3, 3] -> [4, 3] (South)
    - Position: [4, 3] -> [4, 4] (East)

Array: ['South', 'East', 'South', 'East', 'East', 'East', 'South', 'East']

#####################

Example 2:

User Prompt:
Starting position: [2, 4]
Goal position: [1, 1]
Obstacles: [[1, 2], [2, 2], [2, 3], [3, 2], [3, 3], [4, 3]]

Output:
    - Position: [2, 4] -> [1, 4] (North)
    - Position: [1, 4] -> [0, 4] (North)
    - Position: [0, 4] -> [0, 3] (West)
    - Position: [0, 3] -> [0, 2] (West)
    - Position: [0, 2] -> [0, 1] (West)
    - Position: [0, 1] -> [1, 1] (South)

Array: ['North', 'North', 'West', 'West', 'West', 'South']

"""

In [68]:
def ask_slm(prompt):
    print("Asking SLM...")
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )

    return response["message"]["content"]

In [69]:
for grid_info in grids[:5]:
    grid = grid_info["grid"]
    start = grid_info["start"]
    goal = grid_info["goal"]
    obstacles = grid_info["obstacles"]

    print(f"Grid:\n{'\n'.join(' '.join(row) for row in grid)}")
    prompt = f"Starting position: {start}\nGoal position: {goal}\nObstacles: {obstacles}"
    print(f"Prompt:\n{prompt}\n")

    response = ask_slm(prompt)
    print(f"Response:\n{response}\n")

Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .
Prompt:
Starting position: [0, 0]
Goal position: [2, 1]
Obstacles: [[1, 0], [0, 4], [3, 4], [4, 3]]

Asking SLM...
Response:
 - Position: [0, 0] -> [0, 1] (East)
- Position: [0, 1] -> [1, 1] (North)
- Position: [1, 1] -> [1, 2] (East)
- Position: [1, 2] -> [2, 2] (North)
- Position: [2, 2] -> [2, 3] (East)
- Position: [2, 3] -> [2, 4] (North)
- Position: [2, 4] -> [3, 4] (East)
- Position: [3, 4] -> [3, 3] (South)
- Position: [3, 3] -> [3, 2] (West)
- Position: [3, 2] -> [2, 2] (South)
- Position: [2, 2] -> [2, 1] (West)

Array: ['East', 'North', 'East', 'North', 'East', 'North', 'East', 'South', 'West', 'South', 'West']

Grid:
. . B . S
. . B B .
. . . . .
G . . B .
. . . . .
Prompt:
Starting position: [0, 4]
Goal position: [3, 0]
Obstacles: [[0, 2], [1, 2], [1, 3], [3, 3]]

Asking SLM...
Response:
 - Position: [0, 4] -> [0, 3] (West)
- Position: [0, 3] -> [0, 2] (West)
- Position: [0, 2] -> [0, 1] (West)
- Position: [0, 1] -> [0

In [ ]:
user_prompt = f"Grid:\n{'\n'.join(' '.join(row) for row in grid_info['grid'])}\nStarting position: {grid_info['start']}\nGoal position: {grid_info['goal']}."

In [ ]:
print(user_prompt)

Grid:
. B B . S
. . B . .
. . . . G
. . . . .
B . . . B
Starting position: [0, 4]
Goal position: [2, 4].


In [ ]:
ask_slm(user_prompt)

Asking SLM...


" ['North', 'North', 'North', 'North']"

In [60]:
SYSTEM_PROMPT = """
You are a grid world navigation engine. 

A robot is on a 5x5 coordinate plane where (0,0) is top-left and (4,4) is bottom-right.
Movement rules:
- North: (x-1, y)
- South: (x+1, y)
- East:  (x, y+1)
- West:  (x, y-1)

CRITICAL: Do not output any Python code, BFS algorithms, or introductory conversational text. You must output ONLY a valid JSON array of strings representing the path directions.

Input Format:
Starting position: [x, y]
Goal position: [x, y]
Blocked cells: [[x, y], [x, y]...]
Open cells: All cells not blocked.
"""

In [61]:
ask_slm("""
The robot is at position [2, 2] in a 5×5 grid.

Coordinates are [row, column]. Row indices increase from top to bottom, and column indices increase from left to right.

North decreases the row by 1.
South increases the row by 1.
East increases the column by 1.
West decreases the column by 1.

Question: The robot executes the action North. What is its new position?

Return only the new position.
""")

Asking SLM...


' ["2", "1"]'

In [62]:
ask_slm("""
The robot is at position [2, 2] in a 5×5 grid.

Coordinates are [row, column]. Row indices increase from top to bottom, and column indices increase from left to right.

North decreases the row by 1.
South increases the row by 1.
East increases the column by 1.
West decreases the column by 1.

Question: The robot executes the action South. What is its new position?

Return only the new position.
""")

Asking SLM...


' ["2", "3"]'

In [63]:
ask_slm("""
The robot is at position [2, 2] in a 5×5 grid.

Coordinates are [row, column]. Row indices increase from top to bottom, and column indices increase from left to right.

North decreases the row by 1.
South increases the row by 1.
East increases the column by 1.
West decreases the column by 1.

Question: The robot executes the action East. What is its new position?

Return only the new position.
""")

Asking SLM...


' [3, 2]'

In [52]:
ask_slm("""
The robot is at position [2, 2] in a 5×5 grid.

Coordinates are [row, column]. Row indices increase from top to bottom, and column indices increase from left to right.

North decreases the row by 1.
South increases the row by 1.
East increases the column by 1.
West decreases the column by 1.

Question: The robot executes the action West. What is its new position?

Return only the new position.
""")

Asking SLM...


' The new position of the robot after executing the action West is [2, 1].'

In [53]:
ask_slm("""
The robot is at position [2, 2] in a 5×5 grid.

Coordinates are [row, column]. Row indices increase from top to bottom, and column indices increase from left to right.

North decreases the row by 1.
South increases the row by 1.
East increases the column by 1.
West decreases the column by 1.

Question: The robot executes the action North. What is its new position?

Return only the new position.
""")

Asking SLM...


' The new position of the robot after executing the action North would be [2, 1].'